<a href="https://colab.research.google.com/github/Decoding-Data-Science/aiguild/blob/main/30july_of_dds_hr_chatbot_free.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DDS HR Enterprise Chatbot — GitHub Models (GPT-5)
Free GitHub key → GPT-5 (reasoning model) + embeddings → your HR PDFs → Gradio chat. No OpenAI key, no credit card.

In [ ]:
!pip install -q llama_index llama-index-readers-file llama-index-llms-openai-like gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 99.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.5/164.5 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 10.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cpu requires setuptools<82, but you have setuptools 83.0.0 which is incompatible.


## Step 1 — Put your HR PDFs in a `data` folder
In the Colab file browser (folder icon, left sidebar), create a folder called `data` and upload your HR PDFs into it.

## Step 2 — Connect to GitHub Models
One free key, kept in Colab Secrets. Add it before running: key icon 🔑 in the left sidebar → Add new secret → name it `GITHUB_TOKEN` → turn on Notebook access.

In [ ]:
from google.colab import userdata

GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
GITHUB_MODELS_BASE_URL = "https://models.github.ai/inference"

## Step 3 — Load the documents

In [ ]:
from llama_index.core import SimpleDirectoryReader
from llama_index.readers.file import PDFReader

documents = SimpleDirectoryReader(
    input_dir="data",
    required_exts=[".pdf"],
    file_extractor={".pdf": PDFReader()}
).load_data()

print(len(documents))
print(documents[0].text[:500])

13
DDS Employee Handbook (Synthetic) v1
Effective date: March 03, 2026  Dubai (GST)
Note: This document is a synthetic, training-friendly employee handbook for demos, onboarding
simulations, and HR-policy chatbot prototypes. It is not legal advice and must be reviewed by qualified
counsel before any real-world use.
1. Welcome to Decoding Data Science (DDS)
DDS is a Dubai-based academy, consulting practice, and community focused on data science, AI, and
applied generative AI. We operate with a glob


## Step 4 — Configure GPT-5 + embeddings
**Note:** GPT-5 is a reasoning model on GitHub Models — no `temperature` parameter. `OpenAILike` is used instead of the plain `OpenAI` class because llama_index's `OpenAI` class only recognizes bare OpenAI model names (`gpt-5`), not GitHub's prefixed ones (`openai/gpt-5`). `model_name=` (not `model=`) is used for the embedding model for the same reason.

In [ ]:
from llama_index.core import VectorStoreIndex, Settings
from llama_index.llms.openai_like import OpenAILike
from llama_index.embeddings.openai import OpenAIEmbedding

Settings.llm = OpenAILike(
    model="openai/gpt-4o-mini",
    api_base=GITHUB_MODELS_BASE_URL,
    api_key=GITHUB_TOKEN,
    is_chat_model=True,
    context_window=128000,
    temperature=0.2,   # gpt-4o-mini isn't a reasoning model, so temperature works here
)

Settings.embed_model = OpenAIEmbedding(
    model_name="openai/text-embedding-3-small",
    api_base=GITHUB_MODELS_BASE_URL,
    api_key=GITHUB_TOKEN,
)

Settings.chunk_size = 600
Settings.chunk_overlap = 200

## Step 5 — Ayesha, the DDS HR system prompt

In [ ]:
system_prompt = '''

You are Ayesha, the Decoding Data Science (DDS) Enterprise HR Chatbot. Your objective is to interact politely and professionally with employees, answering only HR-related questions. Use only information directly from the connected HR documents to provide your answers. Always provide an explicit citation indicating the document source for every answer. Do not offer information or suggestions beyond what is present in these documents.

If the requested information cannot be found in the connected documents, politely instruct the user to email connect@decodingdatascience.com for further assistance. For questions outside of HR (such as food, restaurants, or non-work matters), inform the user that you can only answer HR-related questions. If a question is unclear or possibly HR-related but ambiguous, persist in seeking clarification or ask the user to rephrase. Never attempt to answer non-HR, personal, or unrelated questions.

- Remain polite and professional in all interactions.
- Respond exclusively to HR-related topics (e.g., payroll, benefits, time-off, HR policies, leave, compliance, hiring, employee development) using only the connected HR documents.
- Always include an explicit citation to the relevant document(s) for every answer.
- If a question is off-topic, state that you can only address HR questions.
- If you are unsure whether the question is HR-related, politely ask for clarification or rephrasing.
- For HR-related questions where the answer is not in the connected documents, kindly direct the user to email connect@decodingdatascience.com.
- Use a friendly and formal tone.
- Always persist in seeking clarification for ambiguous or insufficiently clear questions before providing an answer or deferring.

**Output Format:**
Respond in short, clear paragraphs (2-4 sentences). Do not use markdown or code blocks. Every answer must include the citation showing which document(s) the information is sourced from.

# Examples

**Example 1**
Input: What is the process for applying for maternity leave?
Output: Thank you for your question regarding maternity leave. To apply for maternity leave, please fill out the leave request form available on the HR portal and submit it to your manager for approval. Source: Document 2, Leave Policy.

**Example 2**
Input: Where is the best place to get lunch nearby?
Output: I can only assist with HR-related questions. If you have a question about HR policies, benefits, or company leave, please let me know!

**Example 3**
Input: How can I apply for a sabbatical leave?
Output: I am sorry, but I could not find information regarding sabbatical leave in our HR documents. For further assistance, please email connect@decodingdatascience.com.

'''

## Step 6 — Build the index and try a question

In [ ]:
index = VectorStoreIndex.from_documents(documents=documents)
query_engine = index.as_query_engine(system_prompt=system_prompt)

response = query_engine.query("What are DDS standard office hours in Dubai?")
print(response)

DDS standard office hours in Dubai are from 9:00 AM to 6:00 PM, Monday to Friday.


## Step 7 — Gradio interface

In [ ]:
import gradio as gr

def query_document(query):
    response = query_engine.query(query)
    return str(response)

interface = gr.Interface(
    fn=query_document,
    inputs=gr.Textbox(label="Enter your query", placeholder="Type your question here..."),
    outputs=gr.Textbox(label="Response"),
    title="DDS Enterprise HR Chatbot — Ayesha (GPT-5)",
    description="Ask questions about the HR documents loaded into the system. Powered by GitHub Models — free, one key, no credit card."
)

if __name__ == "__main__":
    interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6d24380e0e1fb9df8b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
